In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns




In [ ]:
df=pd.read_csv("credit_risk_dataset.csv")
df.head()


# EXPLORATORY DATA ANALYSIS(EDA)

In [ ]:
print(df.shape)
print(df.info())

In [ ]:
#missing value

df.isnull().sum()

In [ ]:
#target balance
#tells us balanced and unbalanced
df["loan_status"].value_counts()

In [ ]:
sns.countplot(x=df["loan_status"],data=df)

In [ ]:
#outlier check
numerical_cols=df.select_dtypes(include=np.number).columns

for i in numerical_cols:
    sns.boxplot(x=df[i])
    plt.show()



In [ ]:
(df["person_age"]>100).sum()

In [ ]:
(df["person_emp_length"]>60).sum()

In [ ]:
df.describe()

# correaltion check

In [ ]:
sns.heatmap(df.corr(numeric_only=True),annot=True)

In [ ]:
df_copy=df.copy()

In [ ]:
df_copy

In [ ]:
#remove duplicated rows
df_copy.drop_duplicates(inplace=True)
print(f"rows after removing duplicated:{df_copy.shape[0]}")

In [ ]:
#remove ages below 18 or below 100
df_copy=df_copy[df_copy["person_age"]>=18]
df_copy=df_copy[df_copy["person_age"]<=100]

In [ ]:
#remove employent lenghth greater than age or extreame outlier(>60)
df_copy["person_emp_length"]<=df_copy["person_age"]
df_copy=df_copy[df_copy["person_emp_length"]<=60]

In [ ]:
#remove the rows with zero and negative
df_copy=df_copy[df_copy["loan_amnt"]>0]
df_copy.info()

# 4...feature target  @ train test split

In [ ]:
x=df_copy.drop("loan_status",axis=1)
y=df_copy["loan_status"]

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,stratify=y)

In [ ]:
df_copy.select_dtypes(include=[str]).columns

In [ ]:
numerical_cols=['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
       'loan_int_rate', 'loan_status', 'loan_percent_income',
       'cb_person_cred_hist_length']
categorical_cols=['person_home_ownership', 'loan_intent', 'loan_grade',
       'cb_person_default_on_file']

In [ ]:
df_copy.info()

# 5.handle classs imbalance  

In [ ]:
# 0->no loan

neg,pos=np.bincount(y_train)
scaler_weight=neg/pos


In [ ]:
scaler_weight

# data preprocessing   pipelines & column transformer

In [ ]:
#logistic regression: impute+scaler numercial,impute+  one -hot-encoding

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder


preprocessor_lr=ColumnTransformer(
    transformers=[
        #pipeline-for numerical cols
        ("num",Pipeline([
            ("imputer",SimpleImputer(strategy="median")),
            ("scaler",StandardScaler())
         ]),numerical_cols),
         
         #pipeline-- for categorical cols
         ("cat",Pipeline([
            ("imputer",SimpleImputer(strategy="constant",fill_value="Missing")),
            ("onehot",OneHotEncoder(handle_unknown="ignore"))
             
         ]),categorical_cols)
    ]
)

In [ ]:
# XGboost : impute numerical only(no scaling)
preprocessor_xg=ColumnTransformer(
    transformers=[
        #pipeline --- for numerical cols
        ("num",Pipeline([
            ("imputer",SimpleImputer(strategy="median"))
        ]),numerical_cols),
        #pipeline-- for categorical cols.
        ("cat",Pipeline([
            ("imputer",SimpleImputer(strategy="constant",fill_value="Missing")),
            ("onehot",OneHotEncoder(handle_unknown="ignore"))


        ]),categorical_cols)
    ])
preprocessor_xg

# cross validation


In [ ]:
from sklearn.model_selection import StratifiedKFold,cross_validate
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

scoring={"roc_auc":'roc_auc',"accuracy":"accuracy","precision":"precision","recall":"recall","f1":"f1"}

In [ ]:
#logisticregression

from sklearn.linear_model import LogisticRegression
lr_cv_pipeline=Pipeline([
    ("preproccer","preprocessor_lr"),
    ("classifier",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=42))
])

In [ ]:
#xg boost
from xgboost import XGBClassifier
xgb_cv_pipeline=Pipeline([
    ("preprocess",preprocessor_xg),
    ("classification",XGBClassifier(
    n_estimator=300,max_depth=5,learnig_rate=0.1,scale_pos_weight=scaler_weight,random_state=42
    ))
])

In [65]:
for cv_name,pipe in [("logistic regression",lr_cv_pipeline),
                     ("xgboost",xgb_cv_pipeline)]:

                    cv_results=cross_validate(pipe,x_train,y_train,cv=cv,scoring=scoring,n_jobs=-1)

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Sundram\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_validation.py", line 856, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Sundram\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Sundram\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2k

In [64]:
cv_results

NameError: name 'cv_results' is not defined